In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_err = pd.read_csv("error_logs.csv")

# Helper to convert timestamp ints to timezone-aware datetimes (UTC->Asia/Shanghai)
def to_tz_dt(s):
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)

# Aggregate function for generic dataframe with specified name column
def aggregate_kpi(df, name_col):
    agg = df.groupby(['cmdb_id', name_col]).agg(
        count_points = ('value', 'count'),
        min_timestamp = ('timestamp', 'min'),
        max_timestamp = ('timestamp', 'max'),
        mean_value = ('value', 'mean'),
        p95_value = ('value', lambda x: x.quantile(0.95)),
        p5_value  = ('value', lambda x: x.quantile(0.05))
    ).reset_index()
    # Standardize KPI name column label
    agg = agg.rename(columns={name_col: 'KPI_name'})
    # Convert min/max timestamps to timezone-aware datetimes (Asia/Shanghai)
    agg['min_timestamp'] = to_tz_dt(agg['min_timestamp'])
    agg['max_timestamp'] = to_tz_dt(agg['max_timestamp'])
    # Keep numeric columns rounded for compactness
    agg['mean_value'] = agg['mean_value'].round(6)
    agg['p95_value'] = agg['p95_value'].round(6)
    agg['p5_value'] = agg['p5_value'].round(6)
    return agg

# Produce full group-level tables (stored for later steps)
metric_agg_full = aggregate_kpi(df_metric, 'kpi_name')
trace_agg_full  = aggregate_kpi(df_trace, 'trace_name')
log_agg_full    = aggregate_kpi(df_log, 'log_name')

# Top 20 by count for each
metric_top20 = metric_agg_full.sort_values('count_points', ascending=False).head(20)
trace_top20  = trace_agg_full.sort_values('count_points', ascending=False).head(20)
log_top20    = log_agg_full.sort_values('count_points', ascending=False).head(20)

# Error logs aggregation: per cmdb_id count and earliest timestamp
error_agg_full = df_err.groupby('cmdb_id').agg(
    count_of_error_rows = ('timestamp', 'count'),
    earliest_timestamp = ('timestamp', 'min')
).reset_index()
error_agg_full['earliest_timestamp'] = to_tz_dt(error_agg_full['earliest_timestamp'])
error_top20 = error_agg_full.sort_values('count_of_error_rows', ascending=False).head(20)

# Display compact results (top 20 for each)
metric_top20, trace_top20, log_top20, error_top20

```
Out[1]:
```
summary = (
    "Summary of aggregated telemetry (top groups shown):\n\n"
    "Metrics: Top group is shippingservice2-0 (kpi: mesh.source.shippingservice2.jaeger-collector...) "
    "with 30 points (2022-03-21 02:30 → 02:59 +08), mean=830.825, P95=1243.75. "
    "Many adservice KPIs also appear in the top groups (each with 30 points, same time range) "
    "covering grpc/http counts, latencies and JVM/runtime metrics.\n\n"
    "Traces: Top trace groups are for adservice-0 (trace.from_frontend.*) and shippingservice2-0 "
    "(trace.from_frontend2-0.*), and several frontend-1 outgoing traces. All top groups have 30 points "
    "(2022-03-21 02:30 → 02:59 +08). Example: adservice-0 trace row_count mean ≈4.87 (P95=7.0).\n\n"
    "Logs: Top log groups include adservice-0 log.row_count (30 points, mean≈386.87, P95≈475.3) "
    "and cartservice-0/1/2 log.row_count (means ≈279–286). Most log.error_count groups in the top 20 "
    "are zero or near zero.\n\n"
    "Error logs: Frontend instances show the most error rows: frontend-2 has 230 error rows "
    "(earliest 2022-03-21 02:30:06 +08), frontend-0 has 180 (02:30:10 +08), frontend-1 has 166 (02:30:06 +08). "
    "Cartservice-1 has 27 error rows; cartservice-0 and cartservice-2 have 4 and 2 respectively.\n\n"
    "Overall: top metric/trace/log groups all span the same 30-minute window (02:30–02:59 +08). "
    "Key operational signals: high metric values for shippingservice2-0, heavy request/row counts in adservice and frontend traces, "
    "and concentrated error-log activity on frontend pods (frontend-2, frontend-0, frontend-1)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           KPI_name  count_points             min_timestamp             max_timestamp    mean_value     p95_value      p5_value
4885  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    830.825000   1243.750000    288.550000
0              adservice                                     app.grpc.count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00     73.966667     92.100000     44.850000
1              adservice                                       app.grpc.mrt            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00      0.002465      0.002580      0.002323
2              adservice                                        app.grpc.rr            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    100.000000    100.000000    100.000000
3              adservice                                        app.grpc.sr            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00     76.696766     79.322344     73.933287
4              adservice                                     app.http.count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00     24.000000     24.000000     24.000000
5              adservice                                       app.http.mrt            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00      0.009452      0.009727      0.009171
6              adservice                                        app.http.rr            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    100.000000    100.000000    100.000000
7              adservice                                        app.http.sr            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    100.000000    100.000000    100.000000
8              adservice    runtime.java_lang_ClassLoading_LoadedClassCount            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00   4990.000000   4990.000000   4990.000000
9              adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00   4990.000000   4990.000000   4990.000000
10             adservice  runtime.java_lang_ClassLoading_UnloadedClassCount            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00      0.000000      0.000000      0.000000
11             adservice             runtime.java_lang_ClassLoading_Verbose            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00      0.000000      0.000000      0.000000
12             adservice  runtime.java_lang_Compilation_CompilationTimeM...            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00      1.000000      1.000000      1.000000
13             adservice  runtime.java_lang_Compilation_TotalCompilation...            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00  43673.000000  43673.000000  43673.000000
14             adservice  runtime.java_lang_GarbageCollector_CollectionC...            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00  41244.816667  41356.091667  41133.741667
15             adservice  runtime.java_lang_GarbageCollector_CollectionC...            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00      3.000000      3.000000      3.000000
16             adservice  runtime.java_lang_GarbageCollector_CollectionT...            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00  64789.858333  64963.112500  64616.954167
17             adservice  runtime.java_lang_GarbageCollector_CollectionT...            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    177.000000    177.000000    177.000000
18             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00      1.000000      1.000000      1.000000,                  cmdb_id                               KPI_name  count_points             min_timestamp             max_timestamp  mean_value  p95_value   p5_value
7            adservice-0        trace.from_frontend-1.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    4.866667   7.000000   3.000000
6            adservice-0       trace.from_frontend-1.error_rate            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000   0.000000   0.000000
5            adservice-0     trace.from_frontend-1.duration_p95            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000021   0.000030   0.000014
4            adservice-0    trace.from_frontend-1.duration_mean            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000015   0.000018   0.000012
3            adservice-0        trace.from_frontend-0.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    5.166667   7.550000   2.450000
2            adservice-0       trace.from_frontend-0.error_rate            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000   0.000000   0.000000
1            adservice-0     trace.from_frontend-0.duration_p95            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000023   0.000069   0.000013
0            adservice-0    trace.from_frontend-0.duration_mean            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000016   0.000027   0.000012
15           adservice-1        trace.from_frontend-0.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    5.200000   7.000000   3.450000
14           adservice-1       trace.from_frontend-0.error_rate            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000   0.000000   0.000000
1244  shippingservice2-0   trace.from_frontend2-0.duration_mean            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000069   0.000103   0.000058
1245  shippingservice2-0    trace.from_frontend2-0.duration_p95            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000103   0.000272   0.000062
1246  shippingservice2-0      trace.from_frontend2-0.error_rate            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000   0.000000   0.000000
1247  shippingservice2-0       trace.from_frontend2-0.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    7.500000  13.000000   2.450000
726           frontend-1  trace.to_currencyservice-1.error_rate            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000   0.000000   0.000000
727           frontend-1   trace.to_currencyservice-1.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00   23.766667  34.550000  13.000000
704           frontend-1   trace.to_cartservice-2.duration_mean            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.047112   0.244875   0.000000
705           frontend-1    trace.to_cartservice-2.duration_p95            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.095482   0.694460   0.000000
706           frontend-1      trace.to_cartservice-2.error_rate            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000   0.000000   0.000000
707           frontend-1       trace.to_cartservice-2.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    6.900000   9.000000   4.000000,                     cmdb_id         KPI_name  count_points             min_timestamp             max_timestamp  mean_value  p95_value  p5_value
0               adservice-0  log.error_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000       0.00      0.00
1               adservice-0    log.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00  386.866667     475.30    232.90
2               adservice-1  log.error_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000       0.00      0.00
3               adservice-1    log.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00   34.066667      42.00     20.90
4               adservice-2  log.error_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000       0.00      0.00
5               adservice-2    log.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00   34.200000      42.00     21.80
6             cartservice-0  log.error_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.133333       0.55      0.00
7             cartservice-0    log.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00  278.733333     322.60    198.50
8             cartservice-1  log.error_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.900000       7.05      0.00
9             cartservice-1    log.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00  285.633333     329.00    251.00
10            cartservice-2  log.error_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.066667       0.00      0.00
11            cartservice-2    log.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00  278.466667     311.55    190.20
20        currencyservice-1  log.error_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000       0.00      0.00
21        currencyservice-1    log.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00  238.366667     309.85    129.65
23        currencyservice-2    log.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00  238.900000     308.50    130.85
22        currencyservice-2  log.error_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000       0.00      0.00
19        currencyservice-0    log.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00  238.100000     308.00    125.55
18        currencyservice-0  log.error_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000       0.00      0.00
50  recommendationservice-1  log.error_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00    0.000000       0.00      0.00
51  recommendationservice-1    log.row_count            30 2022-03-21 02:30:00+08:00 2022-03-21 02:59:00+08:00   72.433333      84.55     43.40,          cmdb_id  count_of_error_rows        earliest_timestamp
5     frontend-2                  230 2022-03-21 02:30:06+08:00
3     frontend-0                  180 2022-03-21 02:30:10+08:00
4     frontend-1                  166 2022-03-21 02:30:06+08:00
1  cartservice-1                   27 2022-03-21 02:32:51+08:00
0  cartservice-0                    4 2022-03-21 02:34:46+08:00
2  cartservice-2                    2 2022-03-21 02:33:06+08:00)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

tz = pytz.timezone('Asia/Shanghai')

def to_tz_dt_series(s):
    """Convert epoch seconds series to timezone-aware datetimes in Asia/Shanghai."""
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (before any filtering)
metric_global_p95 = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Prepare metric rows within the incident window (UTC+8)
start_window = pd.Timestamp("2022-03-21 02:30:00").tz_localize(tz)
end_window   = pd.Timestamp("2022-03-21 02:59:00").tz_localize(tz)

# Add timezone-aware timestamp column for filtering and later use
df_metric['ts_tz'] = to_tz_dt_series(df_metric['timestamp'])

# Filter rows in window (inclusive)
mask_window = (df_metric['ts_tz'] >= start_window) & (df_metric['ts_tz'] <= end_window)
df_metric_window = df_metric[mask_window].copy()

# 3) Join global P95 onto the windowed rows for comparison
df_metric_window = df_metric_window.merge(metric_global_p95, on=['cmdb_id','kpi_name'], how='left')

# 4) Identify anomaly rows where value > global_p95 (ensure global_p95 exists)
# If global_p95 is NaN (no global series), treat as non-anomalous
df_metric_window['is_anomaly'] = (df_metric_window['global_p95'].notna()) & (df_metric_window['value'] > df_metric_window['global_p95'])

# Store full per-row anomaly results for later steps
metric_anomalies_full = df_metric_window[df_metric_window['is_anomaly']].copy()

# 5) Aggregate required summary grouped by (cmdb_id, kpi_name)
# points_in_window: count of rows in window per group (from df_metric_window)
points_in_window = (
    df_metric_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(points_in_window=('value','count'),
         max_value_in_window=('value','max'))
)

# anomaly_count and earliest_anomaly_timestamp from anomalies
anomaly_groups = (
    metric_anomalies_full
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(anomaly_count=('value','count'),
         earliest_anomaly_timestamp=('ts_tz','min'))
)

# Merge summaries and global_p95
summary = (
    points_in_window
    .merge(anomaly_groups, on=['cmdb_id','kpi_name'], how='left')
    .merge(metric_global_p95, on=['cmdb_id','kpi_name'], how='left')
)

# Replace NaN anomaly_count with 0 and keep only groups with anomaly_count > 0
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
summary = summary[summary['anomaly_count'] > 0].copy()

# For groups where earliest_anomaly_timestamp may be NaT, keep as-is (shouldn't happen after filter)
# Compute max_breach_ratio = max_value_in_window / global_p95 (guard divide-by-zero)
summary['max_breach_ratio'] = np.where(
    summary['global_p95'] > 0,
    summary['max_value_in_window'] / summary['global_p95'],
    np.nan
)

# Round numeric columns for compactness
summary['global_p95'] = summary['global_p95'].round(6)
summary['max_value_in_window'] = summary['max_value_in_window'].round(6)
summary['max_breach_ratio'] = summary['max_breach_ratio'].round(6)

# Keep earliest_anomaly_timestamp in Asia/Shanghai tz (already in ts_tz)
# Select and order columns as requested
summary = summary[[
    'cmdb_id',
    'kpi_name',
    'points_in_window',
    'anomaly_count',
    'earliest_anomaly_timestamp',
    'max_value_in_window',
    'global_p95',
    'max_breach_ratio'
]]

# Sort by anomaly_count desc and limit to top 20
metric_anomaly_summary_top20 = summary.sort_values('anomaly_count', ascending=False).head(20)

# Display compact result (top 20) and keep full anomalies in metric_anomalies_full
metric_anomaly_summary_top20, metric_anomalies_full.shape

```
Out[2]:
```
summary = (
    "I computed global P95 per (cmdb_id, kpi_name) across the full series, then looked for rows "
    "inside the incident window 2022-03-21 02:30 → 02:59 (+08) where value > global P95.\n\n"
    "Key findings:\n"
    "- Total anomaly rows detected in the window: 3167.\n"
    "- Top aggregated groups (top 20 by anomaly count) mostly have anomaly_count = 2 and points_in_window = 30.\n"
    "- Notable breaches by ratio (examples from the top results):\n"
    "  • shippingservice2-0 (mesh.source...jaeger-collector...): 2 anomalies, earliest 02:39, "
    "max=13.0 vs global P95≈12.275 (breach ratio ≈1.059).\n"
    "  • adservice app.grpc.count: 2 anomalies, earliest 02:47, max=98.0 vs global P95≈92.10 (ratio ≈1.064).\n"
    "  • adservice app.grpc.mrt: 2 anomalies, earliest 02:34, max≈0.002721 vs global P95≈0.00258 (ratio ≈1.054).\n"
    "- Many other top groups are JVM/GC/memory metrics that breach P95 by tiny margins (ratios ≈1.000–1.03).\n\n"
    "Overall conclusion:\n"
    "- Anomalies are widespread but low in count per KPI (mostly 2 points) and generally modest in magnitude "
    "relative to global P95. The most pronounced relative breaches in the top groups occur in adservice and "
    "shippingservice2-0 metrics; many runtime/GC metrics show only marginal exceedances of their P95."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_p95  max_breach_ratio
4884  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                30              2  2022-03-21 02:39:00+08:00         1.300000e+01  1.227500e+01          1.059063
0              adservice                                     app.grpc.count                30              2  2022-03-21 02:47:00+08:00         9.800000e+01  9.210000e+01          1.064061
1              adservice                                       app.grpc.mrt                30              2  2022-03-21 02:34:00+08:00         2.721000e-03  2.580000e-03          1.054410
3              adservice                                        app.grpc.sr                30              2  2022-03-21 02:34:00+08:00         8.076923e+01  7.932234e+01          1.018241
5              adservice                                       app.http.mrt                30              2  2022-03-21 02:42:00+08:00         9.783000e-03  9.727000e-03          1.005772
14             adservice  runtime.java_lang_GarbageCollector_CollectionC...                30              2  2022-03-21 02:58:00+08:00         4.136850e+04  4.135609e+04          1.000300
16             adservice  runtime.java_lang_GarbageCollector_CollectionT...                30              2  2022-03-21 02:58:00+08:00         6.498267e+04  6.496311e+04          1.000301
22             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                30              2  2022-03-21 02:58:00+08:00         2.796855e+08  2.795983e+08          1.000312
24             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                30              2  2022-03-21 02:58:00+08:00         4.136850e+04  4.135609e+04          1.000300
70             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                30              2  2022-03-21 02:41:00+08:00         1.766260e+05  1.708297e+05          1.033930
72             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                30              2  2022-03-21 02:58:00+08:00         1.657011e+07  1.657007e+07          1.000002
118            adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                30              2  2022-03-21 02:44:00+08:00         2.392233e+05  2.249993e+05          1.063218
120            adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                30              2  2022-03-21 02:58:00+08:00         1.657011e+07  1.657007e+07          1.000002
122            adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                30              2  2022-03-21 02:58:00+08:00         2.796855e+08  2.795983e+08          1.000312
153            adservice  runtime.java_lang_MemoryPool_CollectionUsage_u...                30              2  2022-03-21 02:41:00+08:00         2.312973e+05  2.265396e+05          1.021002
178            adservice  runtime.java_lang_MemoryPool_PeakUsage_used.Te...                30              2  2022-03-21 02:58:00+08:00         1.657011e+07  1.657007e+07          1.000002
217            adservice  runtime.java_lang_MemoryPool_Usage_used.Eden_S...                30              2  2022-03-21 02:34:00+08:00         5.182311e+06  5.138524e+06          1.008521
219            adservice  runtime.java_lang_MemoryPool_Usage_used.Surviv...                30              2  2022-03-21 02:41:00+08:00         2.312973e+05  2.265396e+05          1.021002
220            adservice  runtime.java_lang_MemoryPool_Usage_used.Tenure...                30              2  2022-03-21 02:58:00+08:00         1.657011e+07  1.657007e+07          1.000002
230            adservice      runtime.java_lang_Memory_HeapMemoryUsage_used                30              2  2022-03-21 02:36:00+08:00         2.187237e+07  2.174006e+07          1.006086, (3167, 7))```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse variables from previous steps: metric_anomalies_full, df_metric_window, metric_global_p95
# If any are missing, load/compute minimally (but they should exist in the kernel state).
try:
    metric_anomalies_full
except NameError:
    # fallback: load metric.csv and recompute anomalies (shouldn't be necessary in normal flow)
    df_metric = pd.read_csv("metric.csv")
    def to_tz_dt_series(s): return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)
    df_metric['ts_tz'] = to_tz_dt_series(df_metric['timestamp'])
    metric_global_p95 = df_metric.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})
    start_window = pd.Timestamp("2022-03-21 02:30:00").tz_localize(tz)
    end_window   = pd.Timestamp("2022-03-21 02:59:00").tz_localize(tz)
    mask_window = (df_metric['ts_tz'] >= start_window) & (df_metric['ts_tz'] <= end_window)
    df_metric_window = df_metric[mask_window].copy()
    df_metric_window = df_metric_window.merge(metric_global_p95, on=['cmdb_id','kpi_name'], how='left')
    df_metric_window['is_anomaly'] = (df_metric_window['global_p95'].notna()) & (df_metric_window['value'] > df_metric_window['global_p95'])
    metric_anomalies_full = df_metric_window[df_metric_window['is_anomaly']].copy()

# Filter anomalies for cmdb_id starting with 'frontend'
metric_frontend_anomalies_full = metric_anomalies_full[metric_anomalies_full['cmdb_id'].str.startswith('frontend')].copy()

# Store full filtered anomalies for later steps (kernel variable already assigned)
# Compute per-group anomaly aggregations
grp = metric_frontend_anomalies_full.groupby(['cmdb_id','kpi_name'])

anomaly_summary = grp.agg(
    anomaly_count = ('value','count'),
    earliest_anomaly_timestamp = ('ts_tz','min'),
).reset_index()

# Compute longest consecutive anomaly streak per group (in minutes) and earliest timestamp of that longest streak
def compute_longest_streak(ts_series):
    # ts_series: timezone-aware datetime Series, already sorted maybe; ensure sorted ascending
    s = ts_series.sort_values().reset_index(drop=True)
    if s.empty:
        return 0, pd.NaT
    # compute diff in seconds between consecutive timestamps
    diffs = s.diff().dt.total_seconds().fillna(9999999)
    # start a new run when diff != 60 seconds (1 minute)
    new_run = (diffs != 60).cumsum()
    runs = s.groupby(new_run)
    # compute run sizes and earliest timestamp per run
    run_sizes = runs.size()
    max_size = int(run_sizes.max())
    # find earliest timestamp of runs that have max_size
    max_run_id = run_sizes[run_sizes == max_size].index[0]
    # get the timestamps for that run id (group keys are new_run values starting from 1)
    run_idx = max_run_id - 1  # because cumsum started at 1 for first element; offset to locate positions
    # get earliest timestamp by selecting groups where (diffs != 60).cumsum() == max_run_id
    earliest_ts = s[(diffs != 60).cumsum() == max_run_id].iloc[0]
    return max_size, earliest_ts

# Apply per group
streak_results = []
for (cid,kpi), sub in grp:
    length, start_ts = compute_longest_streak(sub['ts_tz'])
    streak_results.append({'cmdb_id': cid, 'kpi_name': kpi, 'longest_streak_length': int(length), 'longest_streak_start': start_ts})

streak_df = pd.DataFrame(streak_results)

# Points in window and max_value_in_window and global_p95 come from full window df_metric_window
# Ensure df_metric_window and metric_global_p95 exist
try:
    df_metric_window
except NameError:
    # reload minimal window data if missing (unlikely)
    df_metric = pd.read_csv("metric.csv")
    def to_tz_dt_series(s): return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)
    df_metric['ts_tz'] = to_tz_dt_series(df_metric['timestamp'])
    start_window = pd.Timestamp("2022-03-21 02:30:00").tz_localize(tz)
    end_window   = pd.Timestamp("2022-03-21 02:59:00").tz_localize(tz)
    mask_window = (df_metric['ts_tz'] >= start_window) & (df_metric['ts_tz'] <= end_window)
    df_metric_window = df_metric[mask_window].copy()

# recompute per-group window stats
window_stats = (
    df_metric_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        points_in_window=('value','count'),
        max_value_in_window=('value','max')
    )
)

# ensure metric_global_p95 exists
try:
    metric_global_p95
except NameError:
    metric_global_p95 = df_metric.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

# Merge all pieces
result = anomaly_summary.merge(window_stats, on=['cmdb_id','kpi_name'], how='left') \
                        .merge(metric_global_p95, on=['cmdb_id','kpi_name'], how='left') \
                        .merge(streak_df, on=['cmdb_id','kpi_name'], how='left')

# Compute max_breach_ratio
result['max_breach_ratio'] = np.where(
    result['global_p95'] > 0,
    result['max_value_in_window'] / result['global_p95'],
    np.nan
)

# Round numeric columns for compactness
result['global_p95'] = result['global_p95'].round(6)
result['max_value_in_window'] = result['max_value_in_window'].round(6)
result['max_breach_ratio'] = result['max_breach_ratio'].round(6)

# Select and order columns per request
result = result[[
    'cmdb_id',
    'kpi_name',
    'points_in_window',
    'anomaly_count',
    'earliest_anomaly_timestamp',
    'max_value_in_window',
    'global_p95',
    'max_breach_ratio',
    'longest_streak_length',
    'longest_streak_start'
]]

# Fill NaNs for display clarity (streaks should exist since groups come from anomalies)
result['longest_streak_length'] = result['longest_streak_length'].fillna(0).astype(int)

# Sort by anomaly_count desc then longest_streak_length desc, limit to top 20
frontend_anomaly_summary_top20 = result.sort_values(['anomaly_count','longest_streak_length'], ascending=[False,False]).head(20)

# Keep full filtered anomaly rows for frontends in kernel variable metric_frontend_anomalies_full (already assigned)
metric_frontend_anomalies_full.shape, frontend_anomaly_summary_top20

```
Out[3]:
```
summary = (
    "Frontend anomaly summary (plain English):\n\n"
    "- Total frontend anomaly rows detected in the incident window: 593.\n"
    "- Anomalies are concentrated on frontend-0 (the top groups shown are all frontend-0).\n"
    "- Common pattern: groups have points_in_window=30 and anomaly_count=2; the longest consecutive anomaly streak\n"
    "  observed per group is 2 minutes (earliest streak timestamps vary per KPI).\n\n"
    "Notable example groups (from top results):\n"
    "  • frontend-0 | container.node-5.container_cpu_cfs_periods — 2 anomalies, earliest 2022-03-21 02:32 (+08),\n"
    "    max=127 vs global P95≈98.7 (breach ratio ≈1.287), longest streak 2 min starting 02:32.\n"
    "  • frontend-0 | mesh.destination.unknown.frontend.* metrics — several groups with 2 anomalies,\n"
    "    examples include a group with max=2247.5 vs P95≈898 (ratio ≈2.50) and others with ratios 1.3–1.8.\n"
    "  • frontend-0 | mesh.source.frontend.adservice.* (request/response/request_bytes) — anomalies with\n"
    "    breach ratios ≈1.29–1.37, earliest around 02:30 (+08).\n\n"
    "Overall conclusion:\n"
    "- Frontend-related telemetry shows widespread but low-frequency breaches (mostly 2 data points per KPI).\n"
    "- The most significant relative breaches (highest ratios) are in mesh-level metrics and some container CPU metrics\n"
    "  for frontend-0. These are good starting points for investigation (mesh request/response volumes and frontend CPU throttling)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

((593, 7),        cmdb_id                                           kpi_name  points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_p95  max_breach_ratio  longest_streak_length      longest_streak_start
2   frontend-0         container.node-5.container_cpu_cfs_periods                30              2  2022-03-21 02:32:00+08:00         1.270000e+02  9.870000e+01          1.286727                      2 2022-03-21 02:32:00+08:00
6   frontend-0   container.node-5.container_fs_usage_MB./dev/vda1                30              2  2022-03-21 02:58:00+08:00         9.752344e+01  9.748379e+01          1.000407                      2 2022-03-21 02:58:00+08:00
7   frontend-0               container.node-5.container_last_seen                30              2  2022-03-21 02:58:00+08:00         1.647803e+09  1.647803e+09          1.000000                      2 2022-03-21 02:58:00+08:00
17  frontend-0  mesh.destination.unknown.frontend.istio_reques...                30              2  2022-03-21 02:35:00+08:00         1.675000e+02  9.212500e+01          1.818182                      2 2022-03-21 02:35:00+08:00
18  frontend-0  mesh.destination.unknown.frontend.istio_reques...                30              2  2022-03-21 02:30:00+08:00         1.453250e+04  1.104500e+04          1.315754                      2 2022-03-21 02:30:00+08:00
20  frontend-0  mesh.destination.unknown.frontend.istio_reques...                30              2  2022-03-21 02:34:00+08:00         2.247500e+03  8.980000e+02          2.502784                      2 2022-03-21 02:34:00+08:00
21  frontend-0  mesh.destination.unknown.frontend.istio_reques...                30              2  2022-03-21 02:35:00+08:00         3.025000e+01  1.663750e+01          1.818182                      2 2022-03-21 02:35:00+08:00
22  frontend-0  mesh.destination.unknown.frontend.istio_reques...                30              2  2022-03-21 02:33:00+08:00         3.494125e+01  2.110394e+01          1.655674                      2 2022-03-21 02:33:00+08:00
24  frontend-0  mesh.destination.unknown.frontend.istio_reques...                30              2  2022-03-21 02:34:00+08:00         3.767500e+01  3.143750e+01          1.198410                      2 2022-03-21 02:34:00+08:00
25  frontend-0  mesh.destination.unknown.frontend.istio_reques...                30              2  2022-03-21 02:35:00+08:00         5.000000e-01  2.750000e-01          1.818182                      2 2022-03-21 02:35:00+08:00
26  frontend-0  mesh.destination.unknown.frontend.istio_reques...                30              2  2022-03-21 02:30:00+08:00         4.100000e+01  3.077500e+01          1.332250                      2 2022-03-21 02:30:00+08:00
28  frontend-0  mesh.destination.unknown.frontend.istio_reques...                30              2  2022-03-21 02:33:00+08:00         6.500000e+00  2.550000e+00          2.549020                      2 2022-03-21 02:33:00+08:00
32  frontend-0  mesh.source.frontend.adservice.istio_request_b...                30              2  2022-03-21 02:30:00+08:00         3.075000e+04  2.314500e+04          1.328581                      2 2022-03-21 02:30:00+08:00
33  frontend-0  mesh.source.frontend.adservice.istio_request_b...                30              2  2022-03-21 02:30:00+08:00         1.530000e+04  1.187875e+04          1.288014                      2 2022-03-21 02:30:00+08:00
34  frontend-0  mesh.source.frontend.adservice.istio_request_d...                30              2  2022-03-21 02:30:00+08:00         6.150000e-02  4.673000e-02          1.316071                      2 2022-03-21 02:30:00+08:00
36  frontend-0  mesh.source.frontend.adservice.istio_request_m...                30              2  2022-03-21 02:30:00+08:00         2.900000e+01  2.292500e+01          1.264995                      2 2022-03-21 02:30:00+08:00
37  frontend-0  mesh.source.frontend.adservice.istio_requests....                30              2  2022-03-21 02:30:00+08:00         1.900000e+01  1.537500e+01          1.235772                      2 2022-03-21 02:30:00+08:00
38  frontend-0  mesh.source.frontend.adservice.istio_requests....                30              2  2022-03-21 02:30:00+08:00         1.000000e+01  7.775000e+00          1.286174                      2 2022-03-21 02:30:00+08:00
39  frontend-0  mesh.source.frontend.adservice.istio_response_...                30              2  2022-03-21 02:30:00+08:00         7.025000e+03  5.146000e+03          1.365138                      2 2022-03-21 02:30:00+08:00
40  frontend-0  mesh.source.frontend.adservice.istio_response_...                30              2  2022-03-21 02:30:00+08:00         1.250000e+03  9.718750e+02          1.286174                      2 2022-03-21 02:30:00+08:00)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing df_trace if present, else load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timezone-aware timestamp column for filtering
def to_tz_dt_series(s):
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)

if 'ts_tz' not in df_trace.columns:
    df_trace['ts_tz'] = to_tz_dt_series(df_trace['timestamp'])

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series (before any filtering)
trace_global_p95 = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Define incident window in UTC+8 and filter rows in that window (inclusive)
start_window = pd.Timestamp("2022-03-21 02:30:00").tz_localize(tz)
end_window   = pd.Timestamp("2022-03-21 02:59:00").tz_localize(tz)
mask_window = (df_trace['ts_tz'] >= start_window) & (df_trace['ts_tz'] <= end_window)
df_trace_window = df_trace[mask_window].copy()

# 3) Join global P95 onto windowed rows (global thresholds computed above)
df_trace_window = df_trace_window.merge(trace_global_p95, on=['cmdb_id','trace_name'], how='left')

# 4) Mark anomalies where value > global_p95 (only when global_p95 exists)
df_trace_window['is_anomaly'] = (df_trace_window['global_p95'].notna()) & (df_trace_window['value'] > df_trace_window['global_p95'])

# Store full per-row trace anomaly results for later steps
trace_anomalies_full = df_trace_window[df_trace_window['is_anomaly']].copy()

# 5) Aggregate required summary per (cmdb_id, trace_name)
points_in_window = (
    df_trace_window
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(points_in_window=('value','count'),
         max_value_in_window=('value','max'))
)

anomaly_groups = (
    trace_anomalies_full
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(anomaly_count=('value','count'),
         earliest_anomaly_timestamp=('ts_tz','min'))
)

summary = (
    points_in_window
    .merge(anomaly_groups, on=['cmdb_id','trace_name'], how='left')
    .merge(trace_global_p95, on=['cmdb_id','trace_name'], how='left')
)

summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
# Keep only groups with anomaly_count > 0 for anomaly-focused table
summary_with_anoms = summary[summary['anomaly_count'] > 0].copy()

# Compute breach ratio
summary_with_anoms['max_breach_ratio'] = np.where(
    summary_with_anoms['global_p95'] > 0,
    summary_with_anoms['max_value_in_window'] / summary_with_anoms['global_p95'],
    np.nan
)

# Round numeric columns for compactness
summary_with_anoms['global_p95'] = summary_with_anoms['global_p95'].round(6)
summary_with_anoms['max_value_in_window'] = summary_with_anoms['max_value_in_window'].round(6)
summary_with_anoms['max_breach_ratio'] = summary_with_anoms['max_breach_ratio'].round(6)

# Select columns for output A (overall top 20 by anomaly_count desc)
agg_cols = [
    'cmdb_id',
    'trace_name',
    'points_in_window',
    'anomaly_count',
    'earliest_anomaly_timestamp',
    'max_value_in_window',
    'global_p95',
    'max_breach_ratio'
]
trace_agg_top20 = summary_with_anoms[agg_cols].sort_values('anomaly_count', ascending=False).head(20)

# 6) Output B: restrict to trace_name containing 'from_frontend' OR cmdb_id starting with 'frontend'
mask_frontend_trace = (
    summary_with_anoms['trace_name'].str.contains('from_frontend', na=False) |
    summary_with_anoms['cmdb_id'].str.startswith('frontend', na=False)
)
trace_frontend_filtered = summary_with_anoms[mask_frontend_trace]
trace_frontend_agg_top20 = trace_frontend_filtered[agg_cols].sort_values(['anomaly_count','points_in_window'], ascending=[False,False]).head(20)

# 7) Total number of trace anomaly rows detected in the window
total_trace_anomaly_rows = int(trace_anomalies_full.shape[0])

# Display compact outputs (use variable names)
trace_agg_top20, trace_frontend_agg_top20, total_trace_anomaly_rows

```
Out[4]:
```
summary = (
    "Trace anomaly summary (window 2022-03-21 02:30 → 02:59 +08):\n\n"
    "- Total trace anomaly rows in the window: 1,230.\n\n"
    "A) Top trace groups overall (by anomaly_count):\n"
    "- shippingservice2-0 | trace.from_frontend2-0.duration_p95: 30 points, 2 anomalies, earliest 02:30, "
    "max=0.000334 vs global P95=0.000272 (ratio ≈1.23).\n"
    "- shippingservice-0 | trace.from_frontend-0.duration_p95: 27 points, 2 anomalies, earliest 02:30, "
    "max=0.000502 vs global P95=0.000142 (ratio ≈3.53) — a large relative spike.\n"
    "- Several shippingservice-0 from_frontend.* groups and recommendationservice2-0 groups also appear in the top list; "
    "most top groups have anomaly_count=2 (low-frequency breaches) but some duration_p95 keys show large breach ratios (up to ~3.5x).\n\n"
    "B) Restricted to traces containing 'from_frontend' or cmdb_id starting with 'frontend':\n"
    "- Many adservice instances (adservice-0, adservice-1, adservice-2 and adservice2-0) show anomalies in from_frontend duration metrics "
    "and row_count. Example: adservice-0 trace.from_frontend-1.duration_p95 has ratio ≈5.04 (strong spike), and other from_frontend duration metrics "
    "show ratios ≈2–3.\n"
    "- These frontend-origin traces also mostly have points_in_window=30 and anomaly_count=2, with earliest anomaly timestamps clustered around 02:30–02:40.\n\n"
    "Overall conclusion:\n"
    "- Trace anomalies are widespread but low in frequency per KPI (mostly 2 anomaly points per group). However, some groups show substantial relative breaches "
    "(notably shippingservice-0 and some adservice from_frontend duration_p95 metrics). Investigate those services and the frontend-origin requests paths "
    "for latency spikes and root-cause correlation with other telemetry (metrics/logs)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                       cmdb_id                                      trace_name  points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95  max_breach_ratio
1245        shippingservice2-0             trace.from_frontend2-0.duration_p95                30              2  2022-03-21 02:30:00+08:00             0.000334    0.000272          1.229071
1181         shippingservice-0              trace.from_frontend-0.duration_p95                27              2  2022-03-21 02:30:00+08:00             0.000502    0.000142          3.525281
1183         shippingservice-0                 trace.from_frontend-0.row_count                27              2  2022-03-21 02:30:00+08:00             4.000000    3.700000          1.081081
1184         shippingservice-0             trace.from_frontend-1.duration_mean                26              2  2022-03-21 02:45:00+08:00             0.000140    0.000102          1.375921
1185         shippingservice-0              trace.from_frontend-1.duration_p95                26              2  2022-03-21 02:43:00+08:00             0.000280    0.000109          2.580645
1188         shippingservice-0             trace.from_frontend-2.duration_mean                28              2  2022-03-21 02:36:00+08:00             0.000157    0.000100          1.576305
1189         shippingservice-0              trace.from_frontend-2.duration_p95                28              2  2022-03-21 02:36:00+08:00             0.000312    0.000165          1.888048
1191         shippingservice-0                 trace.from_frontend-2.row_count                28              2  2022-03-21 02:57:00+08:00             5.000000    4.650000          1.075269
1165  recommendationservice2-0   trace.to_productcatalogservice-2.duration_p95                30              2  2022-03-21 02:43:00+08:00             0.000043    0.000041          1.057811
1180         shippingservice-0             trace.from_frontend-0.duration_mean                27              2  2022-03-21 02:30:00+08:00             0.000296    0.000086          3.457944
1153  recommendationservice2-0                         trace.self.duration_p95                30              2  2022-03-21 02:33:00+08:00             0.003124    0.002993          1.043804
1155  recommendationservice2-0                            trace.self.row_count                30              2  2022-03-21 02:38:00+08:00            35.000000   34.550000          1.013025
1157  recommendationservice2-0   trace.to_productcatalogservice-0.duration_p95                30              2  2022-03-21 02:41:00+08:00             0.000039    0.000037          1.051213
1160  recommendationservice2-0  trace.to_productcatalogservice-1.duration_mean                30              2  2022-03-21 02:47:00+08:00             0.000041    0.000032          1.295419
1161  recommendationservice2-0   trace.to_productcatalogservice-1.duration_p95                30              2  2022-03-21 02:58:00+08:00             0.000109    0.000053          2.048872
1144   recommendationservice-2  trace.to_productcatalogservice-2.duration_mean                30              2  2022-03-21 02:33:00+08:00             0.000048    0.000028          1.742287
1147   recommendationservice-2      trace.to_productcatalogservice-2.row_count                30              2  2022-03-21 02:45:00+08:00             9.000000    8.550000          1.052632
1148  recommendationservice2-0            trace.from_frontend2-0.duration_mean                30              2  2022-03-21 02:40:00+08:00             0.002755    0.002737          1.006540
1149  recommendationservice2-0             trace.from_frontend2-0.duration_p95                30              2  2022-03-21 02:33:00+08:00             0.003442    0.003356          1.025672
1151  recommendationservice2-0                trace.from_frontend2-0.row_count                30              2  2022-03-21 02:38:00+08:00            35.000000   34.550000          1.013025,          cmdb_id                            trace_name  points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  global_p95  max_breach_ratio
1    adservice-0    trace.from_frontend-0.duration_p95                30              2  2022-03-21 02:35:00+08:00             0.000103    0.000069          1.498182
3    adservice-0       trace.from_frontend-0.row_count                30              2  2022-03-21 02:31:00+08:00             9.000000    7.550000          1.192053
4    adservice-0   trace.from_frontend-1.duration_mean                30              2  2022-03-21 02:39:00+08:00             0.000055    0.000018          3.005464
5    adservice-0    trace.from_frontend-1.duration_p95                30              2  2022-03-21 02:39:00+08:00             0.000153    0.000030          5.041186
8    adservice-0   trace.from_frontend-2.duration_mean                30              2  2022-03-21 02:32:00+08:00             0.000060    0.000025          2.439024
9    adservice-0    trace.from_frontend-2.duration_p95                30              2  2022-03-21 02:32:00+08:00             0.000170    0.000053          3.210576
13   adservice-1    trace.from_frontend-0.duration_p95                30              2  2022-03-21 02:41:00+08:00             0.000024    0.000023          1.064302
16   adservice-1   trace.from_frontend-1.duration_mean                30              2  2022-03-21 02:35:00+08:00             0.000017    0.000016          1.055901
17   adservice-1    trace.from_frontend-1.duration_p95                30              2  2022-03-21 02:35:00+08:00             0.000029    0.000024          1.221053
19   adservice-1       trace.from_frontend-1.row_count                30              2  2022-03-21 02:54:00+08:00             7.000000    6.550000          1.068702
20   adservice-1   trace.from_frontend-2.duration_mean                30              2  2022-03-21 02:46:00+08:00             0.000017    0.000016          1.093248
21   adservice-1    trace.from_frontend-2.duration_p95                30              2  2022-03-21 02:46:00+08:00             0.000034    0.000025          1.384929
24   adservice-2   trace.from_frontend-0.duration_mean                30              2  2022-03-21 02:33:00+08:00             0.000015    0.000015          1.030928
28   adservice-2   trace.from_frontend-1.duration_mean                30              2  2022-03-21 02:33:00+08:00             0.000024    0.000018          1.344538
29   adservice-2    trace.from_frontend-1.duration_p95                30              2  2022-03-21 02:33:00+08:00             0.000039    0.000021          1.830986
31   adservice-2       trace.from_frontend-1.row_count                30              2  2022-03-21 02:42:00+08:00             7.000000    6.550000          1.068702
32   adservice-2   trace.from_frontend-2.duration_mean                30              2  2022-03-21 02:34:00+08:00             0.000017    0.000015          1.168385
33   adservice-2    trace.from_frontend-2.duration_p95                30              2  2022-03-21 02:34:00+08:00             0.000027    0.000020          1.343284
36  adservice2-0  trace.from_frontend2-0.duration_mean                30              2  2022-03-21 02:53:00+08:00             0.000030    0.000030          1.015228
37  adservice2-0   trace.from_frontend2-0.duration_p95                30              2  2022-03-21 02:38:00+08:00             0.000051    0.000042          1.221557, 1230)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_err if present, otherwise load
try:
    df_err
except NameError:
    df_err = pd.read_csv("error_logs.csv")

# Ensure timezone-aware timestamp column (UTC -> Asia/Shanghai)
if 'ts_tz' not in df_err.columns:
    df_err['ts_tz'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define window and target frontends
start_window = pd.Timestamp("2022-03-21 02:30:00").tz_localize(tz)
end_window   = pd.Timestamp("2022-03-21 02:59:00").tz_localize(tz)
frontends = ['frontend-0','frontend-1','frontend-2']

# Filter rows
frontend_error_logs_filtered = df_err[
    (df_err['ts_tz'] >= start_window) &
    (df_err['ts_tz'] <= end_window) &
    (df_err['cmdb_id'].isin(frontends))
].copy()

# A) Aggregated table grouped by (cmdb_id, timestamp) showing per-minute error counts
# Use the timezone-aware timestamp as the per-minute key (timestamps are already floored to minute per spec)
aggregated_per_minute = (
    frontend_error_logs_filtered
    .groupby(['cmdb_id', 'ts_tz'], as_index=False)
    .agg(error_count=('message', 'count'))
    .rename(columns={'ts_tz': 'timestamp'})
    .sort_values(['cmdb_id', 'timestamp'], ascending=[True, True])
)

# Keep only first 60 rows for compactness
aggregated_per_minute_head60 = aggregated_per_minute.head(60)

# B) Earliest 30 error log rows across these frontends with selected columns
earliest_30_errors = (
    frontend_error_logs_filtered
    .sort_values('ts_tz', ascending=True)
    .loc[:, ['cmdb_id', 'ts_tz', 'message']]
    .rename(columns={'ts_tz': 'timestamp'})
    .head(30)
)

# Return compact outputs and keep full filtered rows in kernel variable frontend_error_logs_filtered
aggregated_per_minute_head60, earliest_30_errors, frontend_error_logs_filtered.shape

```
Out[5]:
```
summary = (
    "Error-log filter (frontends, 2022-03-21 02:30–02:59 +08):\n\n"
    "- Total filtered error log rows across frontend-0/1/2 in the window: 562 rows.\n\n"
    "A) Per-minute aggregated view (sample shown):\n"
    "- The per-minute counts (first 60 rows) show many single error events recorded for frontend-0 at "
    "fine-grained timestamps between ~02:30 and ~02:37 +08. Errors are distributed by minute and not limited to a single spike.\n\n"
    "B) Earliest 30 error rows:\n"
    "- The earliest errors occur at 02:30:06+08 on frontend-1 and frontend-2, and at 02:30:10+08 on frontend-0.\n"
    "- The log messages in the earliest 30 rows are consistently of the form: \"severity: warning, message: failed to retrieve...\", "
    "indicating repeated warning-level failures early in the window.\n\n"
    "Conclusion:\n"
    "- Frontend pods experienced continuous warning-level errors starting around 02:30+08, with many individual per-minute occurrences "
    "throughout the incident window (562 total rows). The repeated \"failed to retrieve\" warnings across all three frontends are a clear investigation starting point."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                 timestamp  error_count
0   frontend-0 2022-03-21 02:30:10+08:00            1
1   frontend-0 2022-03-21 02:30:23+08:00            1
2   frontend-0 2022-03-21 02:30:27+08:00            1
3   frontend-0 2022-03-21 02:30:39+08:00            1
4   frontend-0 2022-03-21 02:30:42+08:00            1
..         ...                       ...          ...
55  frontend-0 2022-03-21 02:37:24+08:00            1
56  frontend-0 2022-03-21 02:37:31+08:00            1
57  frontend-0 2022-03-21 02:37:42+08:00            1
58  frontend-0 2022-03-21 02:37:48+08:00            1
59  frontend-0 2022-03-21 02:37:58+08:00            1

[60 rows x 3 columns],        cmdb_id                 timestamp                                            message
0   frontend-1 2022-03-21 02:30:06+08:00  severity: warning, message: failed to retrieve...
1   frontend-2 2022-03-21 02:30:06+08:00  severity: warning, message: failed to retrieve...
2   frontend-0 2022-03-21 02:30:10+08:00  severity: warning, message: failed to retrieve...
3   frontend-1 2022-03-21 02:30:16+08:00  severity: warning, message: failed to retrieve...
4   frontend-2 2022-03-21 02:30:17+08:00  severity: warning, message: failed to retrieve...
..         ...                       ...                                                ...
25  frontend-2 2022-03-21 02:31:31+08:00  severity: warning, message: failed to retrieve...
26  frontend-0 2022-03-21 02:31:32+08:00  severity: warning, message: failed to retrieve...
27  frontend-0 2022-03-21 02:31:36+08:00  severity: warning, message: failed to retrieve...
28  frontend-1 2022-03-21 02:31:38+08:00  severity: warning, message: failed to retrieve...
29  frontend-2 2022-03-21 02:31:41+08:00  severity: warning, message: failed to retrieve...

[30 rows x 3 columns], (562, 4))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse stored variables from previous steps:
# metric_anomalies_full, trace_anomalies_full, frontend_error_logs_filtered
# If any are missing, raise informative error (they should exist per prior steps).
for var_name in ('metric_anomalies_full','trace_anomalies_full','frontend_error_logs_filtered'):
    if var_name not in globals():
        raise NameError(f"Required variable '{var_name}' not found in kernel state.")

candidates = [
    'frontend-0','frontend-1','frontend-2',
    'adservice-0','adservice-1','adservice-2',
    'shippingservice-0','shippingservice2-0'
]

# Prepare metric aggregates per cmdb_id
m = metric_anomalies_full.copy()
# compute per-row breach ratio safely
m['breach_ratio'] = np.where(m['global_p95'] > 0, m['value'] / m['global_p95'], np.nan)
metric_agg = (
    m.groupby('cmdb_id', as_index=False)
     .agg(
         earliest_metric_anomaly_timestamp = ('ts_tz', 'min'),
         total_metric_anomaly_count = ('value', 'count'),
         max_metric_breach_ratio = ('breach_ratio', 'max')
     )
)

# Prepare trace aggregates per cmdb_id
t = trace_anomalies_full.copy()
t['breach_ratio'] = np.where(t['global_p95'] > 0, t['value'] / t['global_p95'], np.nan)
trace_agg = (
    t.groupby('cmdb_id', as_index=False)
     .agg(
         earliest_trace_anomaly_timestamp = ('ts_tz', 'min'),
         total_trace_anomaly_count = ('value', 'count'),
         max_trace_breach_ratio = ('breach_ratio', 'max')
     )
)

# Prepare error-log aggregates per cmdb_id (note filtered error logs only include frontends)
e = frontend_error_logs_filtered.copy()
error_agg = (
    e.groupby('cmdb_id', as_index=False)
     .agg(
         earliest_errorlog_timestamp = ('ts_tz', 'min'),
         total_errorlog_count = ('message', 'count')
     )
)

# Build base DataFrame for candidates
comp_df = pd.DataFrame({'cmdb_id': candidates})

# Merge aggregates (left join so missing groups become NaN/0)
comp_df = comp_df.merge(metric_agg, on='cmdb_id', how='left') \
                 .merge(trace_agg, on='cmdb_id', how='left') \
                 .merge(error_agg, on='cmdb_id', how='left')

# Fill NaNs for counts with 0 and ensure integer dtype
comp_df['total_metric_anomaly_count'] = comp_df['total_metric_anomaly_count'].fillna(0).astype(int)
comp_df['total_trace_anomaly_count'] = comp_df['total_trace_anomaly_count'].fillna(0).astype(int)
comp_df['total_errorlog_count'] = comp_df['total_errorlog_count'].fillna(0).astype(int)

# Round breach ratios for compactness
comp_df['max_metric_breach_ratio'] = comp_df['max_metric_breach_ratio'].round(6)
comp_df['max_trace_breach_ratio'] = comp_df['max_trace_breach_ratio'].round(6)

# Compute earliest_of_all timestamp per row (min of the three timestamp columns, ignoring NaT)
ts_cols = ['earliest_metric_anomaly_timestamp','earliest_trace_anomaly_timestamp','earliest_errorlog_timestamp']
comp_df['earliest_of_all'] = comp_df[ts_cols].min(axis=1)

# Sort by earliest_of_all ascending (NaT will be placed last)
comp_df = comp_df.sort_values('earliest_of_all', ascending=True).reset_index(drop=True)

# Select and order columns as requested
component_summary = comp_df[[
    'cmdb_id',
    'earliest_metric_anomaly_timestamp',
    'earliest_trace_anomaly_timestamp',
    'earliest_errorlog_timestamp',
    'total_metric_anomaly_count',
    'total_trace_anomaly_count',
    'total_errorlog_count',
    'max_metric_breach_ratio',
    'max_trace_breach_ratio'
]]

# Keep output compact (8 rows for the 8 candidates)
component_summary

```
Out[6]:
```
component_summary  # reuse existing summary table for clarity
summary_lines = []
for _, row in component_summary.iterrows():
    cm = row['cmdb_id']
    em = str(row['earliest_metric_anomaly_timestamp']) if pd.notna(row['earliest_metric_anomaly_timestamp']) else 'None'
    et = str(row['earliest_trace_anomaly_timestamp']) if pd.notna(row['earliest_trace_anomaly_timestamp']) else 'None'
    ee = str(row['earliest_errorlog_timestamp']) if pd.notna(row['earliest_errorlog_timestamp']) else 'None'
    tm = int(row['total_metric_anomaly_count'])
    tt = int(row['total_trace_anomaly_count'])
    te = int(row['total_errorlog_count'])
    mm = (f"{row['max_metric_breach_ratio']:.6f}" if pd.notna(row['max_metric_breach_ratio']) else 'None')
    mt = (f"{row['max_trace_breach_ratio']:.6f}" if pd.notna(row['max_trace_breach_ratio']) else 'None')
    summary_lines.append(
        f"{cm}: earliest_metric={em}, earliest_trace={et}, earliest_errorlog={ee}; "
        f"metric_count={tm}, trace_count={tt}, errorlog_count={te}; "
        f"max_metric_breach={mm}, max_trace_breach={mt}"
    )

summary = "Per-component compact summary (ordered by earliest event):\n" + "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id earliest_metric_anomaly_timestamp earliest_trace_anomaly_timestamp earliest_errorlog_timestamp  total_metric_anomaly_count  total_trace_anomaly_count  total_errorlog_count  max_metric_breach_ratio  max_trace_breach_ratio
0          frontend-0         2022-03-21 02:30:00+08:00        2022-03-21 02:30:00+08:00   2022-03-21 02:30:10+08:00                         165                        118                   174                 3.976422               16.870461
1          frontend-1         2022-03-21 02:31:00+08:00        2022-03-21 02:30:00+08:00   2022-03-21 02:30:06+08:00                         148                        105                   163                10.354922                5.041186
2   shippingservice-0         2022-03-21 02:31:00+08:00        2022-03-21 02:30:00+08:00                         NaT                          53                         22                     0                 2.000000                3.613054
3  shippingservice2-0         2022-03-21 02:32:00+08:00        2022-03-21 02:30:00+08:00                         NaT                          56                         10                     0                 1.652788                1.500000
4          frontend-2         2022-03-21 02:33:00+08:00        2022-03-21 02:31:00+08:00   2022-03-21 02:30:06+08:00                         138                        114                   225                15.423175                4.498048
5         adservice-0         2022-03-21 02:32:00+08:00        2022-03-21 02:31:00+08:00                         NaT                          44                         14                     0                 1.964208                5.041186
6         adservice-1         2022-03-21 02:36:00+08:00        2022-03-21 02:31:00+08:00                         NaT                          46                         15                     0                 1.699665                1.384929
7         adservice-2         2022-03-21 02:38:00+08:00        2022-03-21 02:31:00+08:00                         NaT                          34                         13                     0                 1.687645                1.830986```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse stored trace anomaly table
try:
    trace_anomalies_full
except NameError:
    raise NameError("trace_anomalies_full not found in kernel. Please run the trace anomaly detection step first.")

# Filter for shippingservice-0 within the incident window (inclusive)
start_window = pd.Timestamp("2022-03-21 02:30:00").tz_localize(tz)
end_window   = pd.Timestamp("2022-03-21 02:59:00").tz_localize(tz)

mask = (
    (trace_anomalies_full['cmdb_id'] == 'shippingservice-0') &
    (trace_anomalies_full['ts_tz'] >= start_window) &
    (trace_anomalies_full['ts_tz'] <= end_window)
)

shippingservice0_trace_anoms_full = trace_anomalies_full[mask].copy()  # stored for later steps

# Compute breach_ratio safely and prepare selected columns
shippingservice0_trace_anoms_full['breach_ratio'] = np.where(
    shippingservice0_trace_anoms_full['global_p95'] > 0,
    shippingservice0_trace_anoms_full['value'] / shippingservice0_trace_anoms_full['global_p95'],
    np.nan
)

# Select and sort, then limit to earliest 20 rows
shippingservice0_trace_anoms_sorted = shippingservice0_trace_anoms_full.sort_values('ts_tz', ascending=True)
shippingservice0_trace_anoms_head = (
    shippingservice0_trace_anoms_sorted
    .loc[:, ['ts_tz','trace_name','value','global_p95','breach_ratio']]
    .rename(columns={'ts_tz': 'timestamp'})
    .head(20)
)

# Round numeric columns for compact display
shippingservice0_trace_anoms_head['value'] = shippingservice0_trace_anoms_head['value'].round(9)
shippingservice0_trace_anoms_head['global_p95'] = shippingservice0_trace_anoms_head['global_p95'].round(9)
shippingservice0_trace_anoms_head['breach_ratio'] = shippingservice0_trace_anoms_head['breach_ratio'].round(6)

# One-line summary: earliest anomaly timestamp, its trace_name, and breach_ratio
if not shippingservice0_trace_anoms_sorted.empty:
    first = shippingservice0_trace_anoms_sorted.iloc[0]
    earliest_ts = first['ts_tz']
    first_trace = first['trace_name']
    first_breach = (first['value'] / first['global_p95']) if (first['global_p95'] and not np.isnan(first['global_p95'])) else np.nan
    shippingservice0_summary = (
        f"Earliest anomaly for shippingservice-0: {earliest_ts} | trace_name: {first_trace} | breach_ratio: {round(first_breach,6)}"
    )
else:
    shippingservice0_summary = "No trace anomalies found for shippingservice-0 in the specified window."

# Return compact table (up to 20 rows) and one-line summary
shippingservice0_trace_anoms_head, shippingservice0_summary

```
Out[7]:
```
Summary for shippingservice-0 trace anomalies (window 2022-03-21 02:30–02:59 +08):

- Earliest anomaly: 2022-03-21 02:30:00+08. The first anomalous trace was trace.from_checkoutservice-0.duration_mean with breach ratio ≈ 1.093.
- Anomalies occur throughout the window (02:30 → 02:58) and affect multiple trace names, mainly from_frontend.* and from_checkoutservice.*.
- Several anomalies show substantial relative spikes vs their global P95, e.g.:
  - 02:32:00 — trace.from_checkoutservice-1.duration_p95, breach_ratio ≈ 3.613
  - 02:58:00 — trace.from_frontend-0.duration_mean, breach_ratio ≈ 3.458
  - 02:45:00 — trace.from_frontend-1.duration_p95, breach_ratio ≈ 2.581
- Overall: shippingservice-0 experienced repeated trace latency/percentile breaches across multiple incoming paths, with the first detected anomaly at 02:30 and the largest relative spikes later in the window.

The original code execution output of IPython Kernel is also provided below for reference:

(                      timestamp                                  trace_name     value  global_p95  breach_ratio
960   2022-03-21 02:30:00+08:00  trace.from_checkoutservice-0.duration_mean  0.000074    0.000068      1.093058
961   2022-03-21 02:30:00+08:00   trace.from_checkoutservice-0.duration_p95  0.000074    0.000068      1.093058
972   2022-03-21 02:30:00+08:00         trace.from_frontend-0.duration_mean  0.000091    0.000086      1.063084
973   2022-03-21 02:30:00+08:00          trace.from_frontend-0.duration_p95  0.000161    0.000142      1.130618
975   2022-03-21 02:30:00+08:00             trace.from_frontend-0.row_count  4.000000    3.700000      1.081081
2924  2022-03-21 02:32:00+08:00  trace.from_checkoutservice-1.duration_mean  0.000190    0.000076      2.500000
2925  2022-03-21 02:32:00+08:00   trace.from_checkoutservice-1.duration_p95  0.000310    0.000086      3.613054
2935  2022-03-21 02:32:00+08:00             trace.from_frontend-0.row_count  4.000000    3.700000      1.081081
6368  2022-03-21 02:36:00+08:00         trace.from_frontend-2.duration_mean  0.000157    0.000100      1.576305
6369  2022-03-21 02:36:00+08:00          trace.from_frontend-2.duration_p95  0.000312    0.000165      1.888048
8385  2022-03-21 02:38:00+08:00          trace.from_frontend-2.duration_p95  0.000209    0.000165      1.264750
8384  2022-03-21 02:38:00+08:00         trace.from_frontend-2.duration_mean  0.000108    0.000100      1.084337
10449 2022-03-21 02:40:00+08:00   trace.from_checkoutservice-2.duration_p95  0.000071    0.000069      1.026012
13345 2022-03-21 02:43:00+08:00          trace.from_frontend-1.duration_p95  0.000110    0.000108      1.013825
15380 2022-03-21 02:45:00+08:00         trace.from_frontend-1.duration_mean  0.000140    0.000102      1.375921
15381 2022-03-21 02:45:00+08:00          trace.from_frontend-1.duration_p95  0.000280    0.000108      2.580645
24560 2022-03-21 02:54:00+08:00         trace.from_frontend-1.duration_mean  0.000104    0.000102      1.022113
25556 2022-03-21 02:55:00+08:00  trace.from_checkoutservice-2.duration_mean  0.000069    0.000065      1.055046
27455 2022-03-21 02:57:00+08:00             trace.from_frontend-2.row_count  5.000000    4.650000      1.075269
28476 2022-03-21 02:58:00+08:00         trace.from_frontend-0.duration_mean  0.000296    0.000086      3.457944, 'Earliest anomaly for shippingservice-0: 2022-03-21 02:30:00+08:00 | trace_name: trace.from_checkoutservice-0.duration_mean | breach_ratio: 1.093058')```
```